# 10 · Actionable Recourse (Counterfactuals)
Beyond *why* an applicant was declined — *what would get them approved*. Minimal, actionable changes over features a borrower can control (never age/gender/location).

In [1]:
# Make `src` importable when running from the notebooks/ folder
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)

In [2]:
import numpy as np
from src.inference import LoanScorer
from src.data_loader import load_raw
from src import recourse
scorer = LoanScorer()
applicants = load_raw().drop(columns=['Loan Status']).sample(300, random_state=3)
proba = scorer.predict_proba(applicants)

In [3]:
# pick a *borderline* declined applicant (just above the threshold)
above = proba[proba >= scorer.threshold]
target_p = above.min() if len(above) else proba.max()
idx = int(np.argmin(np.abs(proba - target_p)))
row = applicants.iloc[[idx]]
print(scorer.explain_text(row))

Decision: DECLINE  (estimated default probability 80.5%, approve below 78.9%)
Main reasons:
  1. Requested / current loan amount: your value 52000.0 vs typical approved 32000.0
  2. Location: your value Plumtree vs typical approved Harare
  3. Number of previous loan defaults: your value 1 vs typical approved 0.0
  4. Declared salary / income: your value 2829.51 vs typical approved 2678.74


In [4]:
rec = recourse.find_recourse(scorer.pipeline, row, scorer.reference, scorer.threshold)
print(recourse.recourse_text(rec))
import json; print(json.dumps(rec, indent=2, default=str))

To reach approval, consider:
  • reduce Requested / current loan amount from 52000.0 to 50000.0
{
  "already_approved": false,
  "default_probability": 0.8049,
  "approved_after_changes": true,
  "changes": [
    {
      "feature": "loan_amount",
      "description": "Requested / current loan amount",
      "from": 52000.0,
      "to": 50000.0,
      "resulting_probability": 0.742
    }
  ]
}


> This is the ethical-AI differentiator. For an optimisation-based version, swap in DiCE (`dice-ml`, Microsoft Research); the native search here keeps the repo dependency-light.